In [1]:
import sys
import pandas as pd
import yfinance as yf

print("Python:", sys.executable)
print("Pronto per scaricare i dati")

Python: c:\Users\gamfe\Documents\cartpole-to-the-market\.venv\Scripts\python.exe
Pronto per scaricare i dati


In [3]:
from time import perf_counter

ticker = "SPY"
inizio = perf_counter()

dati = yf.Ticker(ticker).history(
    start="2010-01-01",
    end="2026-11-15",
    interval="1d",
    auto_adjust=False,
    actions=True
)

secondi_download = perf_counter() - inizio

if dati.empty:
    raise ValueError("Download vuoto: nessun dato ricevuto.")

print("ETF:", ticker)
print("Prima data:", dati.index.min().date())
print("Ultima data:", dati.index.max().date())
print("Numero di righe:", len(dati))
print("Tempo di download:", round(secondi_download, 2), "secondi")

display(dati.head())

ETF: SPY
Prima data: 2010-01-04
Ultima data: 2026-11-13
Numero di righe: 4243
Tempo di download: 0.06 secondi


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,,
2010-01-04 00:00:00-05:00,112.370003,113.389999,111.510002,113.330002,84.368973,118944600,0.0,0.0,0.0
2010-01-05 00:00:00-05:00,113.260002,113.680000,112.849998,113.629997,84.592308,111579900,0.0,0.0,0.0
2010-01-06 00:00:00-05:00,113.519997,113.989998,113.430000,113.709999,84.651871,116074400,0.0,0.0,0.0
2010-01-07 00:00:00-05:00,113.500000,114.330002,113.180000,114.190002,85.009209,131091100,0.0,0.0,0.0
2010-01-08 00:00:00-05:00,113.889999,114.620003,113.660004,114.570000,85.292114,126402800,0.0,0.0,0.0


In [ ]:
inizio_controlli = perf_counter()

prezzi = dati.copy()

prezzi.index = prezzi.index.tz_localize(None).normalize()

print("Date duplicate:", prezzi.index.duplicated().sum())
print("Date in ordine:", prezzi.index.is_monotonic_increasing)

print("\nValori mancanti nelle colonne dei prezzi:")
print(prezzi[["Open", "High", "Low", "Close", "Adj Close"]].isna().sum())

print("\nPrezzi di chiusura non positivi:",
      (prezzi["Close"] <= 0).sum())

print("Prezzi rettificati non positivi:",
      (prezzi["Adj Close"] <= 0).sum())

print("Giorni con frazionamenti registrati:",
      (prezzi["Stock Splits"] != 0).sum())

secondi_controlli = perf_counter() - inizio_controlli
print("\nTempo dei controlli:",
      round(secondi_controlli, 3), "secondi")

Date duplicate: 0
Date in ordine: True

Valori mancanti nelle colonne dei prezzi:
Open         0
High         0
Low          0
Close        0
Adj Close    0
dtype: int64

Prezzi di chiusura non positivi: 0
Prezzi rettificati non positivi: 0
Giorni con frazionamenti registrati: 0

Tempo dei controlli: 0.004 secondi


In [ ]:
import pandas as pd
import yfinance as yf
from time import perf_counter

candidati = ["SPY", "QQQ", "IWM", "EFA", "TLT"]

# Conserviamo i download per usarli nei prossimi controlli
dati_candidati = {}
riepilogo_candidati = []

for simbolo in candidati:
    print("Scarico:", simbolo)
    inizio = perf_counter()

    try:
        tabella = yf.Ticker(simbolo).history(
            start="2010-01-01",
            end="2026-01-01",
            interval="1d",
            auto_adjust=False,
            actions=True
        )

        if tabella.empty:
            raise ValueError("Nessun dato ricevuto")

        dati_candidati[simbolo] = tabella

        riepilogo_candidati.append({
            "ETF": simbolo,
            "Prima data": str(tabella.index.min().date()),
            "Ultima data": str(tabella.index.max().date()),
            "Righe": len(tabella),
            "Date duplicate": int(tabella.index.duplicated().sum()),
            "Chiusure rettificate mancanti": int(
                tabella["Adj Close"].isna().sum()
            ),
            "Secondi": round(perf_counter() - inizio, 2),
            "Esito": "Scaricato"
        })

    except Exception as errore:
        riepilogo_candidati.append({
            "ETF": simbolo,
            "Esito": str(errore)
        })

riepilogo_candidati = pd.DataFrame(riepilogo_candidati)
display(riepilogo_candidati)

In [5]:
import pandas_market_calendars as mcal

# Calendario NYSE come riferimento per le sedute azionarie USA
calendario = mcal.get_calendar("NYSE")

giorni_attesi = calendario.valid_days(
    start_date="2010-01-01",
    end_date="2026-11-14"
).tz_localize(None)

giorni_mancanti = giorni_attesi.difference(prezzi.index)
giorni_extra = prezzi.index.difference(giorni_attesi)

print("Sedute previste dal calendario:", len(giorni_attesi))
print("Giornate presenti nei dati:", len(prezzi))
print("Giornate mancanti:", len(giorni_mancanti))
print("Date fuori calendario:", len(giorni_extra))

if len(giorni_mancanti):
    print("\nDate mancanti:")
    print(giorni_mancanti)

if len(giorni_extra):
    print("\nDate fuori calendario:")
    print(giorni_extra)

Sedute previste dal calendario: 4243
Giornate presenti nei dati: 4243
Giornate mancanti: 0
Date fuori calendario: 0


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import json

# Trova la cartella principale del progetto
cartella = Path.cwd()
root = next(
    p for p in [cartella, *cartella.parents]
    if (p / "pyproject.toml").exists()
)

# Crea una cartella distinta per questo salvataggio
adesso = datetime.now(timezone.utc)
destinazione = (
    root / "results" / "matteo_feasibility"
    / adesso.strftime("%Y%m%dT%H%M%S%fZ")
)
destinazione.mkdir(parents=True, exist_ok=False)

dati.to_csv(destinazione / "SPY_original.csv")
prezzi.to_csv(destinazione / "SPY_checked.csv")

riepilogo = {
    "ticker": ticker,
    "source": "Yahoo Finance via yfinance",
    "yfinance_version": yf.__version__,
    "saved_at_utc": adesso.isoformat(),
    "requested_start": "2010-01-01",
    "requested_end_exclusive": "2026-11-15",
    "first_date": str(prezzi.index.min().date()),
    "last_date": str(prezzi.index.max().date()),
    "rows": len(prezzi),
    "download_seconds": secondi_download,
    "initial_checks_seconds": secondi_controlli,
    "duplicate_dates": int(prezzi.index.duplicated().sum()),
    "missing_price_values": int(
        prezzi[["Open", "High", "Low", "Close", "Adj Close"]]
        .isna().sum().sum()
    ),
    "reference_calendar": "NYSE",
    "missing_sessions": len(giorni_mancanti),
    "extra_dates": len(giorni_extra),
    "adjustment_note": (
        "Provider Adj Close retained; "
        "adjustments not independently verified."
    )
}

with open(destinazione / "summary.json", "w", encoding="utf-8") as f:
    json.dump(riepilogo, f, indent=2)

print("File salvati in:", destinazione)

File salvati in: c:\Users\gamfe\Documents\cartpole-to-the-market\results\matteo_feasibility\20261115T212629605692Z


In [7]:
url_stooq = (
    "https://stooq.com/q/d/l/"
    "?s=spy.us&i=d&d1=20100101&d2=20261114"
)

stooq = pd.read_csv(url_stooq)

if "Date" not in stooq.columns or "Close" not in stooq.columns:
    raise ValueError(
        "Stooq non ha restituito la tabella prevista. "
        f"Risposta ricevuta: {stooq.head()}"
    )

stooq["Date"] = pd.to_datetime(stooq["Date"])
stooq = stooq.set_index("Date").sort_index()

print("Righe scaricate da Stooq:", len(stooq))
print("Prima data:", stooq.index.min())
print("Ultima data:", stooq.index.max())
print("Date duplicate:", stooq.index.duplicated().sum())

display(stooq.tail())

HTTPError: HTTP Error 404: Not Found

In [9]:
import pandas as pd
import yfinance as yf
from time import perf_counter

candidate_etfs = ["SPY", "QQQ", "IWM", "EFA", "TLT"]
start_date = "2010-01-01"
end_date = "2026-01-01"  # Exclusive: includes data through 2025

candidate_data = {}
summary_rows = []

for ticker in candidate_etfs:
    print(f"Downloading {ticker}...")
    started_at = perf_counter()

    try:
        raw_data = yf.Ticker(ticker).history(
            start=start_date,
            end=end_date,
            interval="1d",
            auto_adjust=False,
            actions=True
        )

        if raw_data.empty:
            raise ValueError("No data received")

        candidate_data[ticker] = raw_data

        summary_rows.append({
            "Ticker": ticker,
            "First date": str(raw_data.index.min().date()),
            "Last date": str(raw_data.index.max().date()),
            "Rows": len(raw_data),
            "Duplicate dates": int(
                raw_data.index.duplicated().sum()
            ),
            "Missing adjusted closes": int(
                raw_data["Adj Close"].isna().sum()
            ),
            "Download seconds": round(
                perf_counter() - started_at, 2
            ),
            "Status": "Downloaded"
        })

    except Exception as error:
        summary_rows.append({
            "Ticker": ticker,
            "Status": f"Failed: {error}"
        })

candidate_summary = pd.DataFrame(summary_rows)
display(candidate_summary)

,Ticker,First date,Last date,Rows,Duplicate dates,Missing adjusted closes,Download seconds,Status
0,SPY,2010-01-04,2025-12-31,4024,0,0,4.53,Downloaded
1,QQQ,2010-01-04,2025-12-31,4024,0,0,0.84,Downloaded
2,IWM,2010-01-04,2025-12-31,4024,0,0,1.11,Downloaded
3,EFA,2010-01-04,2025-12-31,4024,0,0,0.86,Downloaded
4,TLT,2010-01-04,2025-12-31,4024,0,0,0.96,Downloaded


In [10]:
import numpy as np
import pandas_market_calendars as mcal

calendar = mcal.get_calendar("NYSE")
expected_dates = calendar.valid_days(
    start_date="2010-01-01",
    end_date="2025-12-31"
).tz_localize(None)

checked_data = {}
quality_rows = []
price_columns = ["Open", "High", "Low", "Close", "Adj Close"]

for ticker, raw_data in candidate_data.items():
    prices = raw_data.copy()
    prices.index = prices.index.tz_localize(None).normalize()

    missing_sessions = expected_dates.difference(prices.index)
    extra_dates = prices.index.difference(expected_dates)

    missing_values = int(prices[price_columns].isna().sum().sum())
    infinite_values = int(
        np.isinf(prices[price_columns].to_numpy()).sum()
    )
    nonpositive_prices = int(
        (prices[price_columns] <= 0).sum().sum()
    )
    duplicate_dates = int(prices.index.duplicated().sum())
    dates_sorted = prices.index.is_monotonic_increasing

    passed = (
        len(missing_sessions) == 0
        and len(extra_dates) == 0
        and missing_values == 0
        and infinite_values == 0
        and nonpositive_prices == 0
        and duplicate_dates == 0
        and dates_sorted
    )

    checked_data[ticker] = prices

    quality_rows.append({
        "Ticker": ticker,
        "Missing sessions": len(missing_sessions),
        "Extra dates": len(extra_dates),
        "Duplicate dates": duplicate_dates,
        "Dates sorted": dates_sorted,
        "Missing price values": missing_values,
        "Infinite price values": infinite_values,
        "Nonpositive prices": nonpositive_prices,
        "Status": "PASS" if passed else "REVIEW"
    })

quality_summary = pd.DataFrame(quality_rows)
display(quality_summary)

,Ticker,Missing sessions,Extra dates,Duplicate dates,Dates sorted,Missing price values,Infinite price values,Nonpositive prices,Status
0,SPY,0,0,0,True,0,0,0,PASS
1,QQQ,0,0,0,True,0,0,0,PASS
2,IWM,0,0,0,True,0,0,0,PASS
3,EFA,0,0,0,True,0,0,0,PASS
4,TLT,0,0,0,True,0,0,0,PASS


In [11]:
from pathlib import Path
from datetime import datetime, timezone
import json

# Stop if any candidate is missing or needs review
assert set(candidate_data) == set(candidate_etfs), "Missing ETF downloads"
assert (quality_summary["Status"] == "PASS").all(), "Review failed checks"

# Locate the project root
current_folder = Path.cwd()
project_root = next(
    folder
    for folder in [current_folder, *current_folder.parents]
    if (folder / "pyproject.toml").exists()
)

saved_at = datetime.now(timezone.utc)
output_folder = (
    project_root
    / "results"
    / "matteo_candidates"
    / saved_at.strftime("%Y%m%dT%H%M%S%fZ")
)
output_folder.mkdir(parents=True, exist_ok=False)

# Save each provider download and its date-normalized copy
for ticker in candidate_etfs:
    candidate_data[ticker].to_csv(
        output_folder / f"{ticker}_original.csv"
    )
    checked_data[ticker].to_csv(
        output_folder / f"{ticker}_checked.csv"
    )

# Combine adjusted closes without filling missing values
adjusted_closes = pd.concat(
    {
        ticker: checked_data[ticker]["Adj Close"]
        for ticker in candidate_etfs
    },
    axis=1
).sort_index()

adjusted_closes.index.name = "Date"
adjusted_closes.to_csv(output_folder / "adjusted_closes.csv")
candidate_summary.to_csv(output_folder / "download_summary.csv", index=False)
quality_summary.to_csv(output_folder / "quality_checks.csv", index=False)

metadata = {
    "status": "Candidate dataset; team approval pending",
    "source": "Yahoo Finance via yfinance",
    "yfinance_version": yf.__version__,
    "saved_at_utc": saved_at.isoformat(),
    "tickers": candidate_etfs,
    "start_date": start_date,
    "end_date_exclusive": end_date,
    "frequency": "Daily",
    "reference_calendar": "NYSE",
    "price_field": "Provider Adj Close",
    "bid_ask_or_midprice_included": False,
    "independent_source_cross_check": "Pending",
    "corporate_action_validation": "Pending"
}

with open(output_folder / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved to:", output_folder)
print("Combined table shape:", adjusted_closes.shape)
print("Missing adjusted closes:", int(adjusted_closes.isna().sum().sum()))

Saved to: c:\Users\gamfe\Documents\cartpole-to-the-market\results\matteo_candidates\20261118T135619608034Z
Combined table shape: (4024, 5)
Missing adjusted closes: 0


In [12]:
# Proposed settings: subject to team approval
train_sessions = 756
embargo_sessions = 20

dates = adjusted_closes.index
fold_rows = []

# Start in July 2013 to allow enough training history and embargo
test_starts = pd.date_range(
    start="2013-07-01",
    end="2025-07-01",
    freq="6MS"
)

for fold_number, test_start in enumerate(test_starts, start=1):
    test_end_exclusive = test_start + pd.DateOffset(months=6)

    test_dates = dates[
        (dates >= test_start)
        & (dates < test_end_exclusive)
    ]

    if test_dates.empty:
        continue

    first_test_position = dates.get_loc(test_dates[0])
    train_stop = first_test_position - embargo_sessions
    train_start = train_stop - train_sessions

    if train_start < 0:
        raise ValueError("Insufficient training history")

    train_dates = dates[train_start:train_stop]
    gap_dates = dates[train_stop:first_test_position]

    fold_rows.append({
        "Fold": fold_number,
        "Train start": train_dates[0].date(),
        "Train end": train_dates[-1].date(),
        "Train sessions": len(train_dates),
        "Embargo start": gap_dates[0].date(),
        "Embargo end": gap_dates[-1].date(),
        "Embargo sessions": len(gap_dates),
        "Test start": test_dates[0].date(),
        "Test end": test_dates[-1].date(),
        "Test sessions": len(test_dates)
    })

proposed_folds = pd.DataFrame(fold_rows)

proposed_folds.to_csv(
    output_folder / "proposed_walk_forward_folds.csv",
    index=False
)

print("PROPOSAL — not an approved or frozen protocol")
print("Number of folds:", len(proposed_folds))
print("No models have been trained or evaluated.")
display(proposed_folds)

PROPOSAL — not an approved or frozen protocol
Number of folds: 25
No models have been trained or evaluated.


,Fold,Train start,Train end,Train sessions,Embargo start,Embargo end,Embargo sessions,Test start,Test end,Test sessions
0,1,2010-06-01,2013-05-31,756,2013-06-03,2013-06-28,20,2013-07-01,2013-12-31,128
1,2,2010-12-01,2013-12-02,756,2013-12-03,2013-12-31,20,2014-01-02,2014-06-30,124
2,3,2011-05-31,2014-06-02,756,2014-06-03,2014-06-30,20,2014-07-01,2014-12-31,128
3,4,2011-11-30,2014-12-02,756,2014-12-03,2014-12-31,20,2015-01-02,2015-06-30,124
4,5,2012-05-30,2015-06-02,756,2015-06-03,2015-06-30,20,2015-07-01,2015-12-31,128
5,6,2012-12-03,2015-12-02,756,2015-12-03,2015-12-31,20,2016-01-04,2016-06-30,125
6,7,2013-06-04,2016-06-02,756,2016-06-03,2016-06-30,20,2016-07-01,2016-12-30,127
7,8,2013-12-03,2016-12-01,756,2016-12-02,2016-12-30,20,2017-01-03,2017-06-30,125
8,9,2014-06-04,2017-06-02,756,2017-06-05,2017-06-30,20,2017-07-03,2017-12-29,126
9,10,2014-12-02,2017-11-30,756,2017-12-01,2017-12-29,20,2018-01-02,2018-06-29,125


In [13]:
proposal_config = {
    "status": "Draft for team discussion; not frozen",
    "candidate_etfs": candidate_etfs,
    "data_start": "2010-01-01",
    "data_end_exclusive": "2026-01-01",
    "frequency": "daily",
    "source": "Yahoo Finance via yfinance",
    "price_field": "Adj Close",
    "bid_ask_available": False,
    "midprice_available": False,
    "evaluation_proposal": {
        "method": "rolling walk-forward",
        "train_sessions": 756,
        "test_months": 6,
        "step_months": 6,
        "embargo_sessions": 20,
        "first_test_start": "2013-07-01",
        "last_test_end": "2025-12-31",
        "number_of_folds": len(proposed_folds)
    },
    "pending_decisions": [
        "Team approval of ETFs and sample period",
        "Walk-forward versus the chronological 80/20 suggestion",
        "Observation window and embargo agreement with Francesco",
        "Training budget feasibility with Stefan",
        "Data format and execution timing agreement with Francesco"
    ],
    "pending_checks": [
        "Independent Stooq comparison",
        "Corporate action adjustment validation"
    ]
}

config_path = output_folder / "data_protocol_proposal.json"

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(proposal_config, file, indent=2)

print("Proposal saved to:", config_path)
print("Status: awaiting team discussion and approval")

Proposal saved to: c:\Users\gamfe\Documents\cartpole-to-the-market\results\matteo_candidates\20261118T135619608034Z\data_protocol_proposal.json
Status: awaiting team discussion and approval
